# Origami RL — GRPO Training

Train an LLM to generate FOLD crease patterns using OpenEnv + Unsloth + TRL.

Follows the [2048 OpenEnv notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb) pattern exactly:
1. `launch_openenv()` spawns the origami environment server
2. LLM generates FOLD JSON crease patterns
3. Reward functions call the server via OpenEnv client
4. GRPO updates policy based on relative rewards

## 1. Install Dependencies

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo
!pip install -qqq fastapi uvicorn requests numpy scipy pydantic

## 2. Clone Origami Env + Setup Paths

In [ ]:
# Clone the origami env repo (skip if running locally)
import subprocess, sys, os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/origami_env.git"  # <-- UPDATE THIS before running on Colab
LOCAL_DIR = "origami_env"

if not Path(LOCAL_DIR).exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError(
            "Set REPO_URL to your GitHub repo URL before running on Colab.\n"
            "Example: https://github.com/yourusername/origami_env.git"
        )
    result = subprocess.run(["git", "clone", REPO_URL, LOCAL_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")
    subprocess.run(["pip", "install", "-e", LOCAL_DIR], check=True)

# Detect repo root by looking for client.py
_candidates = [
    Path(LOCAL_DIR),
    Path.cwd(),
    Path.cwd().parent,
]
working_directory = None
for _c in _candidates:
    if (_c / "client.py").exists():
        working_directory = str(_c.absolute())
        break
if working_directory is None:
    raise RuntimeError(
        "Cannot find origami_env repo root (no client.py found).\n"
        "Set working_directory manually."
    )
sys.path.insert(0, working_directory)
print(f"Working directory: {working_directory}")

In [ ]:
# Import OpenEnv client + models (same pattern as 2048 notebook)
from client import OrigamiEnv
from origami_server.models import OrigamiAction, OrigamiObservation, OrigamiState
print("Origami OpenEnv modules loaded.")

## 3. Load Model + QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 768
lora_rank = 8  # Increased from 4 — 3B model can afford higher rank

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    load_in_4bit = True,
    max_seq_length = max_seq_length,
)

## 4. LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Launch OpenEnv Server

In [ ]:
# Launch origami environment server (same pattern as 2048 notebook)
global port
global openenv_process
port = 8000
openenv_process = None
server = "origami_server.app:app"
environment = {
    **os.environ,
    "PYTHONPATH": working_directory,
}

# Augment Unsloth's launch_openenv with our config
import functools
from unsloth import is_port_open, launch_openenv
launch_openenv = functools.partial(
    launch_openenv,
    working_directory = working_directory,
    server = server,
    environment = environment,
    openenv_class = OrigamiEnv,
)

In [ ]:
# Test the connection — reset and inspect
port, openenv_process = launch_openenv(port, openenv_process)
result = openenv_process.reset(task_name="triangle")
print(f"Server running on port {port}")
print(f"Observation: done={result.done}, reward={result.reward}")
print(f"Task: {result.observation.task}")

## 6. Prompt + Dataset

In [ ]:
import requests

TASK_NAME = "triangle"  # "triangle", "half_fold", "quarter_fold", "letter_fold"

# Fetch task params from the server (paper size, description, etc.)
task_info = requests.get(f"http://localhost:{port}/tasks/{TASK_NAME}").json()

PROMPT_TEMPLATE = """You are an origami designer. Generate a FOLD-format crease pattern
that, when folded, produces the target shape described below.

Target: {description}
Paper size: {width} x {height}

Output a JSON object with these exact fields:
- vertices_coords: [[x, y], ...] — 2D positions on the flat paper (0 to {width} for x, 0 to {height} for y)
- edges_vertices: [[v1, v2], ...] — pairs of vertex indices forming edges
- edges_assignment: ["B"|"M"|"V", ...] — B=boundary, M=mountain fold, V=valley fold
- edges_foldAngle: [angle, ...] — fold angles in degrees (V: 180, M: -180, B: 0)

Rules:
- Boundary edges (B) must outline the paper rectangle
- At least one fold crease (M or V) must exist
- All vertex indices must be valid (0 to N-1)

Output ONLY the JSON object wrapped in ```json ... ``` markers."""

prompt = PROMPT_TEMPLATE.format(
    description=task_info["description"],
    width=task_info["paper"]["width"],
    height=task_info["paper"]["height"],
).strip()

# Build dataset — same prompt repeated 1000x (identical to 2048 pattern)
from datasets import Dataset
dataset = Dataset.from_list([{
    "prompt": [{"role": "user", "content": prompt}],
}] * 1000)

print(f"Task: {task_info['name']} — {task_info['description']}")
print(f"Paper: {task_info['paper']['width']} x {task_info['paper']['height']}")
print(f"Difficulty: {task_info['difficulty']}")
print(f"Dataset: {len(dataset)} rows")
print(f"\nPrompt:\n{prompt[:200]}...")

## 7. Reward Functions

Two reward functions (same pattern as 2048 notebook):
- `valid_fold` — local JSON structure check (fast, no server call)
- `shape_match` — calls the origami server via `launch_openenv`, submits the fold, returns similarity × 20

In [ ]:
import json, re

# --- Reward 1: valid_fold (local check, no server needed) ---

def extract_fold_json(response):
    """Extract FOLD JSON from LLM response text.

    Three-pass extraction:
    1. Fenced code block (```json ... ```) — handles well-formatted LLM output
    2. Brace-counting scan — handles JSON embedded in prose (robust to nested braces)
    3. Whole-response parse — handles bare JSON output
    """
    # Pass 1: fenced code block
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", response, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except json.JSONDecodeError: pass

    # Pass 2: brace-counting scan (handles braces in surrounding prose)
    for m in re.finditer(r'\{', response):
        start, depth = m.start(), 0
        for i, ch in enumerate(response[start:]):
            if ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = response[start : start + i + 1]
                    try:
                        data = json.loads(candidate)
                        if isinstance(data, dict) and "vertices_coords" in data:
                            return data
                    except json.JSONDecodeError:
                        break

    # Pass 3: whole response
    try:
        data = json.loads(response.strip())
        if isinstance(data, dict) and "vertices_coords" in data:
            return data
    except (json.JSONDecodeError, ValueError): pass

    return None

def valid_fold(completions, **kwargs):
    """Does the LLM output parse as valid FOLD JSON?
    +1.0 valid, -0.5 parseable but invalid, -2.0 unparseable."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        fold_data = extract_fold_json(response)
        if fold_data is None:
            scores.append(-2.0); continue
        required = {"vertices_coords", "edges_vertices", "edges_assignment"}
        if not required.issubset(fold_data.keys()):
            scores.append(-0.5); continue
        verts = fold_data.get("vertices_coords", [])
        edges = fold_data.get("edges_vertices", [])
        assigns = fold_data.get("edges_assignment", [])
        if len(edges) != len(assigns):
            scores.append(-0.5); continue
        if not any(a in ("M", "V") for a in assigns) or not any(a == "B" for a in assigns):
            scores.append(-0.5); continue
        n = len(verts)
        if not all(0 <= e[0] < n and 0 <= e[1] < n and e[0] != e[1] for e in edges):
            scores.append(-0.5); continue
        scores.append(1.0)
    return scores

# --- Reward 2: shape_match (calls server via launch_openenv) ---

def shape_match(completions, **kwargs):
    """Submit fold to origami server, get shape similarity reward.
    Calls launch_openenv to ensure server is running, then reset + step."""
    global port, openenv_process
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        fold_data = extract_fold_json(response)
        if fold_data is None:
            scores.append(0.0)
            continue
        try:
            port, openenv_process = launch_openenv(port, openenv_process)
            openenv_process.reset(task_name=TASK_NAME)
            result = openenv_process.step(OrigamiAction(fold_data=fold_data))
            reward = result.reward if result.reward is not None else 0.0
            scores.append(reward)
        except TimeoutError:
            scores.append(-1.0)
        except Exception as e:
            scores.append(-2.0)
    return scores

# Quick test
test_good = [[{"content": json.dumps({
    "vertices_coords": [[0,0],[1,0],[1,1],[0,1]],
    "edges_vertices": [[0,1],[1,2],[2,3],[3,0],[0,2]],
    "edges_assignment": ["B","B","B","B","V"],
    "edges_foldAngle": [0,0,0,0,180]
})}]]
test_bad = [[{"content": "not json"}]]
print(f"valid_fold — good: {valid_fold(test_good)}, bad: {valid_fold(test_bad)}")
print(f"shape_match — good: {shape_match(test_good)}")

## 8. GRPO Trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 2e-4,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 8,
    max_prompt_length = 512,
    max_completion_length = max_seq_length - 512,
    max_steps = 600,
    save_steps = 100,
    output_dir = "outputs",
    report_to = "none",
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [valid_fold, shape_match],
    args = training_args,
    train_dataset = dataset,
)

print(f"Trainer ready: {training_args.max_steps} steps, {training_args.num_generations} generations/step")

## 9. Train!

In [ ]:
trainer.train()

## 10. Save the Trained Model

In [ ]:
save_path = f"origami-{TASK_NAME}-lora"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"LoRA adapter saved to {save_path}/")

## 11. Evaluate — Generate & Score

In [ ]:
import numpy as np
FastLanguageModel.for_inference(model)

NUM_EVAL = 8
messages = [{"role": "user", "content": prompt}]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

print(f"Generating {NUM_EVAL} completions (input: {input_ids.shape[1]} tokens)...\n")

eval_completions = []
for i in range(NUM_EVAL):
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_seq_length - 512,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    eval_completions.append([{"content": response}])
    fold = extract_fold_json(response)
    status = f"parsed ({len(fold.get('vertices_coords', []))} verts)" if fold else "UNPARSEABLE"
    print(f"  Sample {i+1}: {status}")

print(f"\nScoring via server...")
vf_scores = valid_fold(eval_completions)
sm_scores = shape_match(eval_completions)
print(f"  valid_fold:  mean={np.mean(vf_scores):.2f}, scores={vf_scores}")
print(f"  shape_match: mean={np.mean(sm_scores):.2f}, scores={sm_scores}")

## 12. Visualize Best Result

In [ ]:
import matplotlib.pyplot as plt
import requests

EDGE_COLORS = {"M": "red", "V": "blue", "B": "black"}
EDGE_STYLES = {"M": "--", "V": ":", "B": "-"}

best_idx = int(np.argmax(sm_scores))
best_fold = extract_fold_json(eval_completions[best_idx][0]["content"])

if best_fold is None or sm_scores[best_idx] <= 0:
    print("No valid completions to visualize.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

    # Generated crease pattern
    ax1.set_title(f"Generated (sample {best_idx+1})\nreward={sm_scores[best_idx]:.1f}", fontsize=10)
    ax1.set_aspect("equal")
    verts = np.array(best_fold["vertices_coords"])
    for e, a in zip(best_fold["edges_vertices"], best_fold["edges_assignment"]):
        v1, v2 = verts[e[0]], verts[e[1]]
        ax1.plot([v1[0], v2[0]], [v1[1], v2[1]],
                 color=EDGE_COLORS.get(a, "gray"),
                 linestyle=EDGE_STYLES.get(a, "-"), linewidth=2)
    ax1.scatter(verts[:, 0], verts[:, 1], c="black", s=20, zorder=5)
    ax1.grid(True, alpha=0.2)

    # Target crease pattern (from server)
    ax2.set_title("Target", fontsize=10)
    ax2.set_aspect("equal")
    port, openenv_process = launch_openenv(port, openenv_process)
    # Get target from server via HTTP
    target_resp = requests.get(f"http://localhost:{port}/tasks/{TASK_NAME}")
    target = target_resp.json()["target_fold"]
    tverts = np.array(target["vertices_coords"])
    for e, a in zip(target["edges_vertices"], target["edges_assignment"]):
        v1, v2 = tverts[e[0]], tverts[e[1]]
        ax2.plot([v1[0], v2[0]], [v1[1], v2[1]],
                 color=EDGE_COLORS.get(a, "gray"),
                 linestyle=EDGE_STYLES.get(a, "-"), linewidth=2)
    ax2.scatter(tverts[:, 0], tverts[:, 1], c="black", s=20, zorder=5)
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()
    print(f"\nBest FOLD JSON:\n{json.dumps(best_fold, indent=2)}")

## 13. Training Logs

In [ ]:
# Extract training logs from the trainer
logs = trainer.state.log_history

# Parse out loss and reward metrics
steps, losses, rewards = [], [], []
for entry in logs:
    if "loss" in entry:
        steps.append(entry.get("step", 0))
        losses.append(entry["loss"])
    if "reward" in entry:
        rewards.append(entry["reward"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if losses:
    ax1.plot(steps[:len(losses)], losses, color="steelblue", linewidth=1, alpha=0.7)
    # Smoothed line
    if len(losses) > 10:
        window = min(20, len(losses) // 5)
        smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax1.plot(steps[window-1:len(smoothed)+window-1], smoothed, color="navy", linewidth=2)
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training Loss")
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, "No loss data", ha="center", va="center", transform=ax1.transAxes)

if rewards:
    ax2.plot(range(len(rewards)), rewards, color="coral", linewidth=1, alpha=0.7)
    if len(rewards) > 10:
        window = min(20, len(rewards) // 5)
        smoothed = np.convolve(rewards, np.ones(window)/window, mode="valid")
        ax2.plot(range(window-1, len(smoothed)+window-1), smoothed, color="darkred", linewidth=2)
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Reward")
    ax2.set_title("Mean Reward")
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "No reward data", ha="center", va="center", transform=ax2.transAxes)

plt.tight_layout()
plt.show()